<a href="https://colab.research.google.com/github/DavidALutz/bird-vocalization-networks/blob/main/BirdNetworkAnalysis_9_28_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Bird Vocalization Network Analysis
# September 28, 2026
# David Lutz and Ava Bixby, Colby-Sawyer College
#
# Analysis of temporal relationships among annotated bird
# vocalizations across northeastern U.S. national parks.
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Bioacoustics_Project")

print(PROJECT_DIR.exists())

True


In [9]:
# ============================================================
# 2. Define Project Paths
# ============================================================

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Bioacoustics_Project")

DATA_DIR = PROJECT_DIR / "data"
ANNOTATION_DIR = DATA_DIR / "annotations"
METADATA_DIR = DATA_DIR / "metadata"

print("Project directory:", PROJECT_DIR)
print("Annotation directory:", ANNOTATION_DIR)
print("Metadata directory:", METADATA_DIR)

print("\nDirectories found:")
print("Project:", PROJECT_DIR.exists())
print("Annotations:", ANNOTATION_DIR.exists())
print("Metadata:", METADATA_DIR.exists())

Project directory: /content/drive/MyDrive/Bioacoustics_Project
Annotation directory: /content/drive/MyDrive/Bioacoustics_Project/data/annotations
Metadata directory: /content/drive/MyDrive/Bioacoustics_Project/data/metadata

Directories found:
Project: True
Annotations: True
Metadata: True


In [10]:
# ============================================================
# 3. Inventory Annotation Files
# ============================================================

annotation_dirs = {
    "DatasetACAD": ANNOTATION_DIR / "DatasetACAD",
    "DatasetMABI": ANNOTATION_DIR / "DatasetMABI",
    "DatasetSIMR": ANNOTATION_DIR / "DatasetSIMR"
}

total_files = 0

for dataset, dataset_dir in annotation_dirs.items():

    # Look for both CSV and TXT annotation files
    files = list(dataset_dir.rglob("*.csv")) + list(dataset_dir.rglob("*.txt"))

    print(f"{dataset}: {len(files)} annotation files")
    total_files += len(files)

print(f"\nTotal: {total_files} annotation files")

DatasetACAD: 396 annotation files
DatasetMABI: 477 annotation files
DatasetSIMR: 429 annotation files

Total: 1302 annotation files


In [11]:
# ============================================================
# 4. Inspect Example Annotation Files
# ============================================================

import pandas as pd

for dataset, dataset_dir in annotation_dirs.items():

    files = sorted(
        list(dataset_dir.rglob("*.csv")) +
        list(dataset_dir.rglob("*.txt"))
    )

    example_file = files[0]

    print("=" * 70)
    print(f"Dataset: {dataset}")
    print(f"File: {example_file.name}")

    # sep=None asks pandas to infer the delimiter
    example = pd.read_csv(
        example_file,
        sep=None,
        engine="python"
    )

    print(f"Rows: {len(example)}")
    print(f"Columns: {len(example.columns)}")
    print("\nColumn names:")
    print(example.columns.tolist())

    print("\nFirst 3 rows:")
    display(example.head(3))

Dataset: DatasetACAD
File: 6001.41.01x.ACAD3002_20220519_050000.txt
Rows: 173
Columns: 10

Column names:
['Selection', 'View', 'Channel', 'Begin Time (s)', 'End Time (s)', 'Low Freq (Hz)', 'High Freq (Hz)', 'Begin File', 'Species', 'Background']

First 3 rows:


,Selection,View,Channel,Begin Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),Begin File,Species,Background
0,147,Spectrogram 1,1,349.910775,351.757508,115.446,2193.474,6001.41.01x.ACAD3002_20220519_050000.flac,AMCR,NaN
1,148,Spectrogram 1,1,324.838552,326.685285,0.000,2308.920,6001.41.01x.ACAD3002_20220519_050000.flac,AMCR,NaN
2,149,Spectrogram 1,1,340.347338,342.137538,0.000,1962.582,6001.41.01x.ACAD3002_20220519_050000.flac,AMCR,NaN


Dataset: DatasetMABI
File: 5001.01.01x.MABI1001_20220520_060000.txt
Rows: 74
Columns: 10

Column names:
['Selection', 'View', 'Channel', 'Begin Time (s)', 'End Time (s)', 'Low Freq (Hz)', 'High Freq (Hz)', 'Begin File', 'Species', 'Background']

First 3 rows:


,Selection,View,Channel,Begin Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),Begin File,Species,Background
0,1,Spectrogram 1,1,12.698013,15.089101,1347.400,9768.400,5001.01.01x.MABI1001_20220520_060000.flac,OVEN,NaN
1,2,Spectrogram 1,1,32.091692,34.560284,2078.028,9928.356,5001.01.01x.MABI1001_20220520_060000.flac,OVEN,NaN
2,3,Spectrogram 1,1,50.483642,52.989922,2021.100,9928.356,5001.01.01x.MABI1001_20220520_060000.flac,OVEN,NaN


Dataset: DatasetSIMR
File: 1701.15.01x.HBEFV180_20220523_050000.Table.1.selections.txt
Rows: 68
Columns: 10

Column names:
['Selection', 'View', 'Channel', 'Begin Time (s)', 'End Time (s)', 'Low Freq (Hz)', 'High Freq (Hz)', 'Begin File', 'Species', 'Background']

First 3 rows:


,Selection,View,Channel,Begin Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),Begin File,Species,Background
0,50,Spectrogram 1,1,235.360354,236.292610,2078.028,5079.624,1701.15.01x.HBEFV180_20220523_050000.flac,BCCH,NaN
1,49,Spectrogram 1,1,195.077591,196.037814,2078.028,5425.962,1701.15.01x.HBEFV180_20220523_050000.flac,BCCH,NaN
2,55,Spectrogram 1,1,93.974003,95.036774,2539.812,6003.192,1701.15.01x.HBEFV180_20220523_050000.flac,BCCH,NaN


In [12]:
# ============================================================
# 5. Define Annotation Import Function
# ============================================================

def read_annotation_file(file_path, dataset_id):
    """
    Read one Raven annotation file and standardize its columns.

    Parameters
    ----------
    file_path : Path
        Path to the Raven annotation file.
    dataset_id : str
        Dataset source: DatasetACAD, DatasetMABI, or DatasetSIMR.

    Returns
    -------
    pandas.DataFrame
        Standardized annotation events.
    """

    # Read Raven table
    df = pd.read_csv(
        file_path,
        sep=None,
        engine="python"
    )

    # Rename columns
    df = df.rename(columns={
        "Selection": "selection",
        "View": "view",
        "Channel": "channel",
        "Begin Time (s)": "begin_time_s",
        "End Time (s)": "end_time_s",
        "Low Freq (Hz)": "low_freq_hz",
        "High Freq (Hz)": "high_freq_hz",
        "Begin File": "recording_filename",
        "Species": "species_code",
        "Background": "background"
    })

    # Preserve provenance
    df["dataset_ID"] = dataset_id
    df["annotation_filename"] = file_path.name

    # Derived event-level variables
    df["duration_s"] = df["end_time_s"] - df["begin_time_s"]
    df["bandwidth_hz"] = df["high_freq_hz"] - df["low_freq_hz"]

    return df

In [13]:
# Test standardized import on one file from each dataset

test_dfs = []

for dataset, dataset_dir in annotation_dirs.items():

    files = sorted(
        list(dataset_dir.rglob("*.csv")) +
        list(dataset_dir.rglob("*.txt"))
    )

    test_df = read_annotation_file(files[0], dataset)

    test_dfs.append(test_df)

    print(
        f"{dataset}: "
        f"{len(test_df)} events, "
        f"{test_df['species_code'].nunique()} species"
    )

display(test_dfs[0].head())

DatasetACAD: 173 events, 12 species
DatasetMABI: 74 events, 6 species
DatasetSIMR: 68 events, 6 species


,selection,view,channel,begin_time_s,end_time_s,low_freq_hz,high_freq_hz,recording_filename,species_code,background,dataset_ID,annotation_filename,duration_s,bandwidth_hz
0,147,Spectrogram 1,1,349.910775,351.757508,115.446,2193.474,6001.41.01x.ACAD3002_20220519_050000.flac,AMCR,NaN,DatasetACAD,6001.41.01x.ACAD3002_20220519_050000.txt,1.846733,2078.028
1,148,Spectrogram 1,1,324.838552,326.685285,0.000,2308.920,6001.41.01x.ACAD3002_20220519_050000.flac,AMCR,NaN,DatasetACAD,6001.41.01x.ACAD3002_20220519_050000.txt,1.846733,2308.920
2,149,Spectrogram 1,1,340.347338,342.137538,0.000,1962.582,6001.41.01x.ACAD3002_20220519_050000.flac,AMCR,NaN,DatasetACAD,6001.41.01x.ACAD3002_20220519_050000.txt,1.790200,1962.582
3,151,Spectrogram 1,1,346.245576,348.224218,346.338,2078.028,6001.41.01x.ACAD3002_20220519_050000.flac,AMCR,NaN,DatasetACAD,6001.41.01x.ACAD3002_20220519_050000.txt,1.978642,1731.690
4,172,Spectrogram 1,1,365.962682,367.602128,0.000,2078.028,6001.41.01x.ACAD3002_20220519_050000.flac,AMCR,NaN,DatasetACAD,6001.41.01x.ACAD3002_20220519_050000.txt,1.639446,2078.028


In [14]:
# ============================================================
# 6. Build Master Event Table
# ============================================================

all_events = []

for dataset, dataset_dir in annotation_dirs.items():

    files = sorted(
        list(dataset_dir.rglob("*.csv")) +
        list(dataset_dir.rglob("*.txt"))
    )

    print(f"Reading {dataset}: {len(files)} files")

    for file_path in files:
        df = read_annotation_file(file_path, dataset)
        all_events.append(df)

# Combine all annotation files into one DataFrame
events = pd.concat(all_events, ignore_index=True)

print("\nImport complete!")
print(f"Total annotation events: {len(events):,}")
print(f"Total annotation files: {events['annotation_filename'].nunique():,}")
print(f"Total recording files: {events['recording_filename'].nunique():,}")
print(f"Total species codes: {events['species_code'].nunique():,}")

Reading DatasetACAD: 396 files
Reading DatasetMABI: 477 files
Reading DatasetSIMR: 429 files


/tmp/ipykernel_868/394205986.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat(all_events, ignore_index=True)



Import complete!
Total annotation events: 184,207
Total annotation files: 1,295
Total recording files: 1,295
Total species codes: 104


In [15]:
# ============================================================
# 7. Initial Dataset Audit
# ============================================================

dataset_summary = (
    events
    .groupby("dataset_ID")
    .agg(
        annotation_files=("annotation_filename", "nunique"),
        recordings=("recording_filename", "nunique"),
        events=("selection", "size"),
        species=("species_code", "nunique")
    )
)

display(dataset_summary)

print("\nMissing values:")
display(events.isna().sum())

print("\nEvent duration summary:")
display(events["duration_s"].describe())

print("\nFrequency bandwidth summary:")
display(events["bandwidth_hz"].describe())

,annotation_files,recordings,events,species
dataset_ID,,,,
DatasetACAD,395,395,43091,67
DatasetMABI,477,477,95205,85
DatasetSIMR,423,423,45911,84



Missing values:


,0
selection,0
view,0
channel,0
begin_time_s,0
end_time_s,0
low_freq_hz,0
high_freq_hz,0
recording_filename,0
species_code,0
background,178317



Event duration summary:


,duration_s
count,184207.000000
mean,3.297960
std,10.390955
min,0.000000
25%,1.149497
50%,1.562500
75%,2.138613
max,598.991835



Frequency bandwidth summary:


,bandwidth_hz
count,184207.000000
mean,4465.948843
std,1781.301249
min,0.000000
25%,3232.488000
50%,4178.937000
75%,5447.300000
max,16000.000000


In [16]:
# ============================================================
# 8. Annotation QC
# ============================================================

# ------------------------------------------------------------
# A. Identify annotation files with zero events
# ------------------------------------------------------------

all_annotation_files = []

for dataset, dataset_dir in annotation_dirs.items():
    files = (
        list(dataset_dir.rglob("*.csv")) +
        list(dataset_dir.rglob("*.txt"))
    )

    for file_path in files:
        all_annotation_files.append({
            "dataset_ID": dataset,
            "annotation_filename": file_path.name
        })

file_inventory = pd.DataFrame(all_annotation_files)

files_with_events = (
    events[["dataset_ID", "annotation_filename"]]
    .drop_duplicates()
)

empty_files = (
    file_inventory
    .merge(
        files_with_events,
        on=["dataset_ID", "annotation_filename"],
        how="left",
        indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
)

print("Annotation files containing zero events:")
display(empty_files)

print(f"Total empty annotation files: {len(empty_files)}")


# ------------------------------------------------------------
# B. Check unusual event durations
# ------------------------------------------------------------

print("\nZero-duration events:")
print((events["duration_s"] == 0).sum())

print("\nEvents longer than 10 seconds:")
print((events["duration_s"] > 10).sum())

print("\nEvents longer than 60 seconds:")
print((events["duration_s"] > 60).sum())

print("\nLongest 20 events:")
display(
    events.nlargest(20, "duration_s")[
        [
            "dataset_ID",
            "annotation_filename",
            "selection",
            "species_code",
            "begin_time_s",
            "end_time_s",
            "duration_s",
            "low_freq_hz",
            "high_freq_hz",
            "background"
        ]
    ]
)


# ------------------------------------------------------------
# C. Check unusual frequency annotations
# ------------------------------------------------------------

print("\nZero-bandwidth events:")
print((events["bandwidth_hz"] == 0).sum())

print("\nFull-spectrum (0–16000 Hz) events:")
print(
    (
        (events["low_freq_hz"] == 0) &
        (events["high_freq_hz"] == 16000)
    ).sum()
)


# ------------------------------------------------------------
# D. Species codes associated with long events
# ------------------------------------------------------------

print("\nSpecies associated with events > 10 seconds:")

display(
    events.loc[events["duration_s"] > 10, "species_code"]
    .value_counts()
    .head(30)
)

Annotation files containing zero events:


,dataset_ID,annotation_filename
70,DatasetACAD,6531.30.01x.ACAD4004_20230627_050000.txt
876,DatasetSIMR,1704.27.01x.HBEFV108_20220523_060001.Table.1.s...
959,DatasetSIMR,2250.02.01x.KAWW5207_20220620_070001.Table.1.s...
1017,DatasetSIMR,2311.15.01x.KAWW5208_20220722_070000.Table.1.s...
1091,DatasetSIMR,2386.27.01x.KAWW5104_20220720_070001.Table.1.s...
1210,DatasetSIMR,2520.25.01x.HBEFV346_20230618_090001.Table.1.s...
1274,DatasetSIMR,2584.47.01x.HBEFV340_20230607_080001.Table.1.s...


Total empty annotation files: 7

Zero-duration events:
1

Events longer than 10 seconds:
6463

Events longer than 60 seconds:
195

Longest 20 events:


,dataset_ID,annotation_filename,selection,species_code,begin_time_s,end_time_s,duration_s,low_freq_hz,high_freq_hz,background
174559,DatasetSIMR,2525.29.01x.HBEFV308_20230523_050000.Table.1.s...,109,REVI,0.772613,599.764447,598.991835,1729.7,5621.6,NaN
163687,DatasetSIMR,2431.01.01x.KAWW5209_20230612_051312.Table.1.s...,1,REVI,0.991263,599.495937,598.504673,640.0,6054.1,NaN
172943,DatasetSIMR,2514.20.01x.KAWW5005_20230616_060000.Table.1.s...,1,REVI,1.384408,599.260753,597.876345,1225.2,4828.8,NaN
158038,DatasetSIMR,2368.20.01x.MABI1101_20220616_050000.Table.1.s...,111,REVI,2.036290,598.991936,596.955646,1600.0,5546.7,NaN
169261,DatasetSIMR,2481.08.01x.HBEFV009_20230523_090001.Table.1.s...,28,BHVI,6.636423,599.798356,593.161933,2255.0,4939.6,NaN
167216,DatasetSIMR,2465.02.01x.HBEFV340_20230625_070000.Table.1.s...,3,REVI,12.869621,599.059140,586.189519,1225.2,5045.0,NaN
156493,DatasetSIMR,2353.14.01x.MABI1010_20220607_080001.Table.1.s...,8,REVI,0.403226,581.270165,580.866939,1511.1,5600.0,1.0
156463,DatasetSIMR,2352.57.01x.MABI1009_20220607_070001.Table.1.s...,1,RBNU,1.250000,572.318553,571.068553,1619.0,3747.7,1.0
169891,DatasetSIMR,2487.10.01x.HBEFV335_20230717_090001.Table.1.s...,2,REVI,0.285618,529.166650,528.881032,1932.9,4402.7,1.0
158750,DatasetSIMR,2373.44.01x.MABI1106_20220616_070001.Table.1.s...,90,REVI,93.286274,599.596774,506.310500,1706.7,4160.0,NaN



Zero-bandwidth events:
3

Full-spectrum (0–16000 Hz) events:
14

Species associated with events > 10 seconds:


,count
species_code,
REVI,5117
BHVI,610
EACH,97
AMCR,80
UNMA,74
????,59
RBNU,53
AMRO,48
BLJA,37


In [17]:
# ============================================================
# 9. Examine Annotation Duration Patterns
# ============================================================

# Duration categories
events["duration_class"] = pd.cut(
    events["duration_s"],
    bins=[-0.001, 2, 5, 10, 30, 60, 300, float("inf")],
    labels=[
        "0-2 s",
        "2-5 s",
        "5-10 s",
        "10-30 s",
        "30-60 s",
        "1-5 min",
        ">5 min"
    ]
)

# Overall distribution
duration_table = (
    events["duration_class"]
    .value_counts(sort=False)
    .to_frame("events")
)

duration_table["percent"] = (
    duration_table["events"] / len(events) * 100
)

print("Overall duration distribution:")
display(duration_table)


# Duration patterns by dataset
print("\nDuration distribution by dataset:")

display(
    pd.crosstab(
        events["dataset_ID"],
        events["duration_class"],
        normalize="index"
    ).mul(100).round(2)
)


# Species with the longest typical annotations
species_duration = (
    events.groupby("species_code")
    .agg(
        n_events=("species_code", "size"),
        median_duration_s=("duration_s", "median"),
        mean_duration_s=("duration_s", "mean"),
        max_duration_s=("duration_s", "max"),
        n_over_10s=("duration_s", lambda x: (x > 10).sum()),
        n_over_60s=("duration_s", lambda x: (x > 60).sum())
    )
    .sort_values("n_over_10s", ascending=False)
)

display(species_duration.head(25))

Overall duration distribution:


,events,percent
duration_class,,
0-2 s,130517,70.853442
2-5 s,43255,23.481735
5-10 s,3972,2.156270
10-30 s,1886,1.023848
30-60 s,4382,2.378846
1-5 min,176,0.095545
>5 min,19,0.010314



Duration distribution by dataset:


duration_class,0-2 s,2-5 s,5-10 s,10-30 s,30-60 s,1-5 min,>5 min
dataset_ID,,,,,,,
DatasetACAD,78.56,16.14,2.85,0.93,1.51,0.01,0.00
DatasetMABI,67.57,26.92,1.61,0.83,3.05,0.02,0.00
DatasetSIMR,70.44,23.24,2.64,1.51,1.80,0.33,0.04


,n_events,median_duration_s,mean_duration_s,max_duration_s,n_over_10s,n_over_60s
species_code,,,,,,
REVI,6724,50.610840,38.627706,598.991835,5117,160
BHVI,1885,2.176506,13.139246,593.161933,610,12
EACH,313,3.448491,15.132017,189.511240,97,10
AMCR,4837,1.916872,2.534027,119.220405,80,2
UNMA,263,2.657034,12.898453,112.826851,74,2
????,588,1.281534,4.697979,121.349348,59,4
RBNU,1601,2.410962,3.592277,571.068553,53,1
AMRO,5353,1.997486,2.361497,59.246200,48,0
BLJA,3774,0.983367,1.528304,54.356127,37,0


In [18]:
# ============================================================
# 10. Inspect Metadata Files
# ============================================================

for item in sorted(METADATA_DIR.iterdir()):
    print(f"{'[DIR]' if item.is_dir() else '[FILE]'} {item.name}")

[FILE] data_dictionary.csv
[FILE] recording_metadata.csv
[FILE] site_metadata.csv
[FILE] species_metadata.csv


In [19]:
# Preview metadata files

for item in sorted(METADATA_DIR.iterdir()):

    if item.suffix.lower() in [".csv", ".txt"]:

        print("\n" + "=" * 70)
        print(item.name)

        try:
            temp = pd.read_csv(
                item,
                sep=None,
                engine="python"
            )

            print(f"Rows: {len(temp):,}")
            print(f"Columns: {len(temp.columns)}")
            print("Column names:")
            print(temp.columns.tolist())

            display(temp.head(3))

        except Exception as e:
            print(f"Could not read: {e}")


data_dictionary.csv
Could not read: 'utf-8' codec can't decode byte 0x93 in position 4247: invalid start byte

recording_metadata.csv
Rows: 1,302
Columns: 12
Column names:
['Dataset_ID', 'ItemID', 'Recording_FileName', 'Annotation_FileName', 'Year', 'Date', 'Time', 'SiteID', 'ParkID', 'Round1_Annotator', 'Round2_Annotator', 'Round3_Annotator']


,Dataset_ID,ItemID,Recording_FileName,Annotation_FileName,Year,Date,Time,SiteID,ParkID,Round1_Annotator,Round2_Annotator,Round3_Annotator
0,DatasetACAD,6001,6001.41.01x.ACAD3002_20220519_050000.flac,6001.41.01x.ACAD3002_20220519_050000.txt,2022,20220519,541,ACAD3002,ACAD,A13_Amateur,A02_Experienced,A01_Experienced
1,DatasetACAD,6021,6021.41.01x.ACAD3005_20220519_050000.flac,6021.41.01x.ACAD3005_20220519_050000.txt,2022,20220519,541,ACAD3005,ACAD,A13_Amateur,A02_Experienced,A01_Experienced
2,DatasetACAD,6022,6022.41.01x.ACAD3005_20220520_050000.flac,6022.41.01x.ACAD3005_20220520_050000.txt,2022,20220520,541,ACAD3005,ACAD,A13_Amateur,A02_Experienced,A01_Experienced



site_metadata.csv
Rows: 104
Columns: 7
Column names:
['ParkID', 'SiteID', 'Latitude', 'Longitude', 'N_Rec_10min', 'Recorded_in_2022', 'Recorded_in_2023']


,ParkID,SiteID,Latitude,Longitude,N_Rec_10min,Recorded_in_2022,Recorded_in_2023
0,ACAD,ACAD3101,44.350905,-68.292071,24,yes,yes
1,ACAD,ACAD3201,44.403570,-68.248136,24,yes,yes
2,ACAD,ACAD3401,44.282847,-68.345701,24,yes,yes



species_metadata.csv
Rows: 101
Columns: 8
Column names:
['AOU_Code', 'IsBird', 'Common_Name', 'Scientific_Name', 'Family', 'N_DatasetACAD', 'N_DatasetMABI', 'N_DatasetSIMR']


,AOU_Code,IsBird,Common_Name,Scientific_Name,Family,N_DatasetACAD,N_DatasetMABI,N_DatasetSIMR
0,????,Maybe,Unknown,NaN,NaN,99,325,164
1,EACH,No,Eastern Chipmunk,Tamias striatus,NaN,9,170,134
2,RESQ,No,Red Squirrel,Tamiasciurus hudsonicus,NaN,59,153,187


In [20]:
# ============================================================
# 11. Build Master Recording and Event Tables
# ============================================================

# ------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------

recordings = pd.read_csv(
    METADATA_DIR / "recording_metadata.csv"
)

sites = pd.read_csv(
    METADATA_DIR / "site_metadata.csv"
)

species = pd.read_csv(
    METADATA_DIR / "species_metadata.csv"
)


# ------------------------------------------------------------
# Standardize recording metadata column names
# ------------------------------------------------------------

recordings = recordings.rename(columns={
    "Dataset_ID": "dataset_ID",
    "ItemID": "itemID",
    "Recording_FileName": "recording_filename",
    "Annotation_FileName": "annotation_filename",
    "Year": "year",
    "Date": "date",
    "Time": "recording_time",
    "SiteID": "siteID",
    "ParkID": "park_ID",
    "Round1_Annotator": "round1_annotator",
    "Round2_Annotator": "round2_annotator",
    "Round3_Annotator": "round3_annotator"
})

sites = sites.rename(columns={
    "ParkID": "park_ID",
    "SiteID": "siteID",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "N_Rec_10min": "n_recordings_10min",
    "Recorded_in_2022": "recorded_in_2022",
    "Recorded_in_2023": "recorded_in_2023"
})

species = species.rename(columns={
    "AOU_Code": "species_code",
    "IsBird": "is_bird",
    "Common_Name": "common_name",
    "Scientific_Name": "scientific_name",
    "Family": "family"
})


# ------------------------------------------------------------
# Add site information to recording table
# ------------------------------------------------------------

recordings = recordings.merge(
    sites,
    on=["park_ID", "siteID"],
    how="left",
    validate="many_to_one"
)

print(f"Master recording table: {len(recordings):,} recordings")
print(f"Sites: {recordings['siteID'].nunique()}")
print(f"Parks: {recordings['park_ID'].nunique()}")

Master recording table: 1,302 recordings
Sites: 104
Parks: 4


In [21]:
# ------------------------------------------------------------
# Add recording and species metadata to event table
# ------------------------------------------------------------

events = events.merge(
    recordings[
        [
            "dataset_ID",
            "itemID",
            "recording_filename",
            "year",
            "date",
            "recording_time",
            "siteID",
            "park_ID",
            "latitude",
            "longitude",
            "round1_annotator",
            "round2_annotator",
            "round3_annotator"
        ]
    ],
    on=["dataset_ID", "recording_filename"],
    how="left",
    validate="many_to_one"
)

events = events.merge(
    species[
        [
            "species_code",
            "is_bird",
            "common_name",
            "scientific_name",
            "family"
        ]
    ],
    on="species_code",
    how="left",
    validate="many_to_one"
)

print(f"Events: {len(events):,}")
print(f"Recordings represented: {events['itemID'].nunique():,}")
print(f"Sites represented: {events['siteID'].nunique()}")
print(f"Parks represented: {events['park_ID'].nunique()}")

print("\nBird classification:")
display(events["is_bird"].value_counts(dropna=False))

Events: 184,207
Recordings represented: 1,287
Sites represented: 104
Parks represented: 4

Bird classification:


,count
is_bird,
Yes,182634
No,982
Maybe,588
NaN,3


In [22]:
# ============================================================
# 12. Species Metadata Match QC
# ============================================================

unmatched_codes = (
    events.loc[events["common_name"].isna(), "species_code"]
    .value_counts()
    .to_frame("n_events")
)

print("Annotation codes not matched to species metadata:")
display(unmatched_codes)

print(
    f"\nEvents without species metadata: "
    f"{events['common_name'].isna().sum():,}"
)

Annotation codes not matched to species metadata:


,n_events
species_code,
HETH,1
WOTH,1
HAWO,1



Events without species metadata: 3


In [23]:
# ============================================================
# 13. Recording Metadata Match QC
# ============================================================

# Events that failed to match recording metadata
unmatched_recordings = (
    events.loc[events["itemID"].isna()]
    .groupby(
        ["dataset_ID", "recording_filename", "annotation_filename"],
        dropna=False
    )
    .size()
    .reset_index(name="n_events")
)

print(f"Event-bearing annotation files without recording metadata: "
      f"{len(unmatched_recordings)}")

print(f"Events affected: {unmatched_recordings['n_events'].sum():,}")

display(unmatched_recordings)

Event-bearing annotation files without recording metadata: 8
Events affected: 809


,dataset_ID,recording_filename,annotation_filename,n_events
0,DatasetSIMR,2207.01.01x.ACAD3606_20220602_090001.flac,2207.01.01x.ACAD3606_20220602_090001.Table.1.s...,127
1,DatasetSIMR,2252.37.01x.KAWW5104_20220621_090001.flac,2252.37.01x.KAWW5104_20220621_090001.Table.1.s...,1
2,DatasetSIMR,2285.01.01x.HBEFV007_20220715_060000.flac,2285.01.01x.HBEFV007_20220715_060000.Table.1.s...,205
3,DatasetSIMR,2485.01.01x.ACAD3201_20230528_082133.flac,2485.01.01x.ACAD3201_20230528_082133.Table.1.s...,53
4,DatasetSIMR,2506.15.01x.HBEFV111_20230622_060000.flac,2506.15.01x.HBEFV111_20230622_060000.Table.1.s...,42
5,DatasetSIMR,2560.46.01x.KAWW5402_20230527_060000.flac,2560.46.01x.KAWW5402_20230527_060000.Table.1.s...,98
6,DatasetSIMR,2570.50.01x.ACAD3201_20230613_070000.flac,2570.50.01x.ACAD3201_20230613_070000.Table.1.s...,131
7,DatasetSIMR,2610.00.01x.HBEFV007_20230624_060000.flac,2610.00.01x.HBEFV007_20230624_060000.Table.1.s...,152


In [24]:
# ============================================================
# 14. Species Code Match QC
# ============================================================

annotation_codes = set(events["species_code"].dropna().unique())
metadata_codes = set(species["species_code"].dropna().unique())

codes_missing_from_metadata = sorted(
    annotation_codes - metadata_codes
)

codes_not_observed = sorted(
    metadata_codes - annotation_codes
)

print("Codes in annotations but absent from species metadata:")
print(codes_missing_from_metadata)

print("\nNumber of annotation codes:", len(annotation_codes))
print("Number of metadata codes:", len(metadata_codes))
print("Number of unmatched annotation codes:",
      len(codes_missing_from_metadata))

print("\nMetadata codes not observed in annotations:")
print(codes_not_observed)

Codes in annotations but absent from species metadata:
[' HETH', 'HAWO ', 'WOTH ']

Number of annotation codes: 104
Number of metadata codes: 101
Number of unmatched annotation codes: 3

Metadata codes not observed in annotations:
[]


In [25]:
# ============================================================
# 15. Diagnose SIMR Recording Filename Mismatches
# ============================================================

# Extract apparent ItemID from unmatched recording filenames
unmatched_recordings["filename_itemID"] = (
    unmatched_recordings["recording_filename"]
    .str.extract(r"^(\d+)")
    .astype(int)
)

problem_itemIDs = unmatched_recordings["filename_itemID"].tolist()

metadata_matches = (
    recordings.loc[
        recordings["itemID"].isin(problem_itemIDs),
        [
            "dataset_ID",
            "itemID",
            "recording_filename",
            "annotation_filename",
            "siteID",
            "park_ID",
            "date",
            "recording_time"
        ]
    ]
    .sort_values("itemID")
)

display(metadata_matches)

print(f"Problem ItemIDs found in metadata: "
      f"{metadata_matches['itemID'].nunique()} of "
      f"{len(problem_itemIDs)}")

,dataset_ID,itemID,recording_filename,annotation_filename,siteID,park_ID,date,recording_time
585,DatasetSIMR,2207,2207.10.01x.ACAD3606_20220602_090001.flac,2207.10.01x.ACAD3606_20220602_090001.Table.1.s...,ACAD3606,ACAD,20220602,910
622,DatasetSIMR,2252,2252.35.01x.KAWW5104_20220621_090001.flac,2252.35.01x.KAWW5104_20220621_090001.Table.1.s...,KAWW5104,KAWW,20220621,935
652,DatasetSIMR,2285,2285.00.01x.HBEFV007_20220715_060000.flac,2285.00.01x.HBEFV007_20220715_060000.Table.1.s...,HBEFV007,HBEF,20220715,600
762,DatasetSIMR,2485,2485.10.01x.ACAD3201_20230528_082133.flac,2485.10.01x.ACAD3201_20230528_082133.Table.1.s...,ACAD3201,ACAD,20230528,832
776,DatasetSIMR,2506,2506.18.01x.HBEFV111_20230622_060000.flac,2506.18.01x.HBEFV111_20230622_060000.Table.1.s...,HBEFV111,HBEF,20230622,618
803,DatasetSIMR,2560,2560.40.01x.KAWW5402_20230527_060000.flac,2560.40.01x.KAWW5402_20230527_060000.Table.1.s...,KAWW5402,KAWW,20230527,640
485,DatasetSIMR,2570,2570.42.01x.ACAD3201_20230613_070000.flac,2570.42.01x.ACAD3201_20230613_070000.Table.1.s...,ACAD3201,ACAD,20230613,742
543,DatasetSIMR,2610,2610.58.01x.HBEFV007_20230624_050000.flac,2610.58.01x.HBEFV007_20230624_050000.Table.1.s...,HBEFV007,HBEF,20230624,558


Problem ItemIDs found in metadata: 8 of 8


In [26]:
# ============================================================
# 16. Clean Species Codes
# ============================================================

# Remove accidental leading/trailing whitespace
events["species_code"] = events["species_code"].str.strip()
species["species_code"] = species["species_code"].str.strip()

print("Unique annotation codes after cleaning:",
      events["species_code"].nunique())

print(
    "Codes still absent from species metadata:",
    sorted(
        set(events["species_code"].unique()) -
        set(species["species_code"].unique())
    )
)

Unique annotation codes after cleaning: 101
Codes still absent from species metadata: []


In [27]:
# ============================================================
# 17. Assign Authoritative Recording ID
# ============================================================

events["itemID_from_annotation"] = (
    events["annotation_filename"]
    .str.extract(r"^(\d+)")[0]
    .astype(int)
)

print("Unique ItemIDs represented by events:",
      events["itemID_from_annotation"].nunique())

print("ItemIDs not found in recording metadata:")

missing_itemIDs = sorted(
    set(events["itemID_from_annotation"]) -
    set(recordings["itemID"])
)

print(missing_itemIDs)

# we can say: The dataset contains 1,302 recording units identified by ItemID; 1,295 contain ≥1 annotated event and 7 contain zero annotated events.

Unique ItemIDs represented by events: 1295
ItemIDs not found in recording metadata:
[]


In [28]:
# ============================================================
# 18. Build Clean Master Event Table
# ============================================================

# Start from the imported annotation events
master_events = events.copy()

# ------------------------------------------------------------
# Clean identifiers
# ------------------------------------------------------------

master_events["species_code"] = (
    master_events["species_code"]
    .astype(str)
    .str.strip()
)

# Preserve the filename recorded inside the Raven annotation
master_events = master_events.rename(columns={
    "recording_filename": "recording_filename_annotation"
})

# Use ItemID parsed from annotation filename as recording key
master_events["itemID"] = (
    master_events["annotation_filename"]
    .str.extract(r"^(\d+)")[0]
    .astype(int)
)

# ------------------------------------------------------------
# Remove metadata columns from our earlier exploratory merge
# so that we can rebuild them cleanly
# ------------------------------------------------------------

metadata_columns = [
    "year",
    "date",
    "recording_time",
    "siteID",
    "park_ID",
    "latitude",
    "longitude",
    "round1_annotator",
    "round2_annotator",
    "round3_annotator",
    "is_bird",
    "common_name",
    "scientific_name",
    "family",
    "itemID_from_annotation"
]

master_events = master_events.drop(
    columns=[c for c in metadata_columns if c in master_events.columns],
    errors="ignore"
)

# ------------------------------------------------------------
# Add authoritative recording metadata using ItemID
# ------------------------------------------------------------

recording_lookup = recordings[
    [
        "itemID",
        "dataset_ID",
        "recording_filename",
        "year",
        "date",
        "recording_time",
        "siteID",
        "park_ID",
        "latitude",
        "longitude",
        "round1_annotator",
        "round2_annotator",
        "round3_annotator"
    ]
].rename(columns={
    "recording_filename": "recording_filename_metadata"
})

# dataset_ID already exists in annotation data, so use ItemID alone
# and verify agreement afterward
master_events = master_events.merge(
    recording_lookup,
    on="itemID",
    how="left",
    validate="many_to_one",
    suffixes=("_annotation_source", "_metadata_source")
)

# Verify dataset provenance agrees
dataset_disagreement = (
    master_events["dataset_ID_annotation_source"] !=
    master_events["dataset_ID_metadata_source"]
).sum()

print("Dataset-ID disagreements:", dataset_disagreement)

# Keep one authoritative dataset ID
master_events["dataset_ID"] = (
    master_events["dataset_ID_metadata_source"]
)

master_events = master_events.drop(columns=[
    "dataset_ID_annotation_source",
    "dataset_ID_metadata_source"
])

# ------------------------------------------------------------
# Add species metadata
# ------------------------------------------------------------

master_events = master_events.merge(
    species[
        [
            "species_code",
            "is_bird",
            "common_name",
            "scientific_name",
            "family"
        ]
    ],
    on="species_code",
    how="left",
    validate="many_to_one"
)

print(f"Master events: {len(master_events):,}")
print(f"Recordings with events: {master_events['itemID'].nunique():,}")
print(f"Species/annotation codes: {master_events['species_code'].nunique()}")
print(f"Missing recording metadata: {master_events['siteID'].isna().sum():,}")
print(f"Missing species metadata: {master_events['is_bird'].isna().sum():,}")

Dataset-ID disagreements: 0
Master events: 184,207
Recordings with events: 1,295
Species/annotation codes: 101
Missing recording metadata: 0
Missing species metadata: 0


In [29]:
# ============================================================
# 19. Create Event IDs and Sort Chronologically
# ============================================================

master_events = master_events.sort_values(
    ["itemID", "begin_time_s", "end_time_s", "selection"]
).reset_index(drop=True)

# Unique event identifier
master_events["event_id"] = (
    master_events["itemID"].astype(str)
    + "_"
    + master_events["selection"].astype(str)
)

print("Total events:", f"{len(master_events):,}")
print("Unique event IDs:", f"{master_events['event_id'].nunique():,}")

duplicate_event_ids = (
    master_events["event_id"]
    .duplicated()
    .sum()
)

print("Duplicate event IDs:", duplicate_event_ids)

Total events: 184,207
Unique event IDs: 184,086
Duplicate event IDs: 121


In [30]:
# ============================================================
# 20. Add Event Counts to Recording Table
# ============================================================

event_counts = (
    master_events
    .groupby("itemID")
    .size()
    .rename("n_events")
)

master_recordings = recordings.copy()

master_recordings = master_recordings.merge(
    event_counts,
    on="itemID",
    how="left"
)

master_recordings["n_events"] = (
    master_recordings["n_events"]
    .fillna(0)
    .astype(int)
)

master_recordings["has_events"] = (
    master_recordings["n_events"] > 0
)

print(f"Total recordings: {len(master_recordings):,}")
print(f"Recordings with events: {master_recordings['has_events'].sum():,}")
print(f"Zero-event recordings: {(~master_recordings['has_events']).sum():,}")

display(
    master_recordings.loc[
        ~master_recordings["has_events"],
        [
            "itemID",
            "dataset_ID",
            "park_ID",
            "siteID",
            "date",
            "recording_time",
            "n_events"
        ]
    ]
)

Total recordings: 1,302
Recordings with events: 1,295
Zero-event recordings: 7


,itemID,dataset_ID,park_ID,siteID,date,recording_time,n_events
61,6531,DatasetACAD,ACAD,ACAD4004,20230627,530,0
429,2520,DatasetSIMR,HBEF,HBEFV346,20230618,925,0
537,2584,DatasetSIMR,HBEF,HBEFV340,20230607,847,0
547,1704,DatasetSIMR,HBEF,HBEFV108,20220523,627,0
620,2250,DatasetSIMR,KAWW,KAWW5207,20220620,702,0
674,2311,DatasetSIMR,KAWW,KAWW5208,20220722,815,0
781,2386,DatasetSIMR,KAWW,KAWW5104,20220720,0,0


In [31]:
# ============================================================
# 21. Investigate Repeated Raven Selection Numbers
# ============================================================

duplicate_mask = master_events.duplicated(
    subset=["itemID", "selection"],
    keep=False
)

repeated_selections = (
    master_events.loc[duplicate_mask]
    .sort_values(
        ["itemID", "selection", "begin_time_s"]
    )
)

print(
    "Events involved in repeated ItemID + Selection combinations:",
    len(repeated_selections)
)

print(
    "Unique repeated ItemID + Selection combinations:",
    repeated_selections[
        ["itemID", "selection"]
    ].drop_duplicates().shape[0]
)

print(
    "Recordings affected:",
    repeated_selections["itemID"].nunique()
)

display(
    repeated_selections[
        [
            "itemID",
            "dataset_ID",
            "park_ID",
            "siteID",
            "annotation_filename",
            "selection",
            "species_code",
            "begin_time_s",
            "end_time_s",
            "low_freq_hz",
            "high_freq_hz"
        ]
    ].head(50)
)

Events involved in repeated ItemID + Selection combinations: 242
Unique repeated ItemID + Selection combinations: 121
Recordings affected: 1


,itemID,dataset_ID,park_ID,siteID,annotation_filename,selection,species_code,begin_time_s,end_time_s,low_freq_hz,high_freq_hz
142605,6017,DatasetACAD,ACAD,ACAD3002,6017.30.01x.ACAD3002_20220625_050000.txt,1,BTNW,9.902633,11.042708,3060.265,5935.06
142606,6017,DatasetACAD,ACAD,ACAD3002,6017.30.01x.ACAD3002_20220625_050000.txt,1,BTNW,9.902633,11.042708,3060.265,5935.06
142771,6017,DatasetACAD,ACAD,ACAD3002,6017.30.01x.ACAD3002_20220625_050000.txt,2,BTNW,306.955395,307.954138,3616.677,6642.90
142772,6017,DatasetACAD,ACAD,ACAD3002,6017.30.01x.ACAD3002_20220625_050000.txt,2,BTNW,306.955395,307.954138,3616.677,6642.90
142785,6017,DatasetACAD,ACAD,ACAD3002,6017.30.01x.ACAD3002_20220625_050000.txt,3,BTNW,331.697845,333.130005,3321.500,6954.30
142786,6017,DatasetACAD,ACAD,ACAD3002,6017.30.01x.ACAD3002_20220625_050000.txt,3,BTNW,331.697845,333.130005,3321.500,6954.30
142781,6017,DatasetACAD,ACAD,ACAD3002,6017.30.01x.ACAD3002_20220625_050000.txt,4,BTNW,323.802120,325.092949,3529.000,6746.70
142782,6017,DatasetACAD,ACAD,ACAD3002,6017.30.01x.ACAD3002_20220625_050000.txt,4,BTNW,323.802120,325.092949,3529.000,6746.70
142819,6017,DatasetACAD,ACAD,ACAD3002,6017.30.01x.ACAD3002_20220625_050000.txt,5,BRCR,465.748841,466.182258,6435.300,8096.00
142820,6017,DatasetACAD,ACAD,ACAD3002,6017.30.01x.ACAD3002_20220625_050000.txt,5,BRCR,465.748841,466.182258,6435.300,8096.00


In [32]:
# ============================================================
# 22. Check for Exact Duplicate Events
# ============================================================

event_definition = [
    "itemID",
    "selection",
    "species_code",
    "begin_time_s",
    "end_time_s",
    "low_freq_hz",
    "high_freq_hz"
]

exact_duplicate_mask = master_events.duplicated(
    subset=event_definition,
    keep=False
)

exact_duplicates = (
    master_events.loc[exact_duplicate_mask]
    .sort_values(
        ["itemID", "selection", "begin_time_s"]
    )
)

print(
    "Rows involved in exact duplicate annotations:",
    len(exact_duplicates)
)

print(
    "Extra rows beyond unique annotations:",
    master_events.duplicated(
        subset=event_definition
    ).sum()
)

display(
    exact_duplicates[
        event_definition + ["annotation_filename"]
    ].head(50)
)

Rows involved in exact duplicate annotations: 242
Extra rows beyond unique annotations: 121


,itemID,selection,species_code,begin_time_s,end_time_s,low_freq_hz,high_freq_hz,annotation_filename
142605,6017,1,BTNW,9.902633,11.042708,3060.265,5935.06,6017.30.01x.ACAD3002_20220625_050000.txt
142606,6017,1,BTNW,9.902633,11.042708,3060.265,5935.06,6017.30.01x.ACAD3002_20220625_050000.txt
142771,6017,2,BTNW,306.955395,307.954138,3616.677,6642.90,6017.30.01x.ACAD3002_20220625_050000.txt
142772,6017,2,BTNW,306.955395,307.954138,3616.677,6642.90,6017.30.01x.ACAD3002_20220625_050000.txt
142785,6017,3,BTNW,331.697845,333.130005,3321.500,6954.30,6017.30.01x.ACAD3002_20220625_050000.txt
142786,6017,3,BTNW,331.697845,333.130005,3321.500,6954.30,6017.30.01x.ACAD3002_20220625_050000.txt
142781,6017,4,BTNW,323.802120,325.092949,3529.000,6746.70,6017.30.01x.ACAD3002_20220625_050000.txt
142782,6017,4,BTNW,323.802120,325.092949,3529.000,6746.70,6017.30.01x.ACAD3002_20220625_050000.txt
142819,6017,5,BRCR,465.748841,466.182258,6435.300,8096.00,6017.30.01x.ACAD3002_20220625_050000.txt
142820,6017,5,BRCR,465.748841,466.182258,6435.300,8096.00,6017.30.01x.ACAD3002_20220625_050000.txt


In [33]:
# ============================================================
# 23. Remove Exact Duplicate Annotation Events
# ============================================================

n_before = len(master_events)

# Remove exact duplicate annotations while retaining
# the first occurrence of each event
master_events = (
    master_events
    .drop_duplicates(
        subset=[
            "itemID",
            "selection",
            "species_code",
            "begin_time_s",
            "end_time_s",
            "low_freq_hz",
            "high_freq_hz"
        ],
        keep="first"
    )
    .copy()
)

n_after = len(master_events)

print(f"Events before duplicate removal: {n_before:,}")
print(f"Events after duplicate removal:  {n_after:,}")
print(f"Duplicate rows removed:          {n_before - n_after:,}")

Events before duplicate removal: 184,207
Events after duplicate removal:  184,086
Duplicate rows removed:          121


In [34]:
# ============================================================
# 24. Create Final Event IDs
# ============================================================

# Sort events chronologically within each recording
master_events = (
    master_events
    .sort_values(
        ["itemID", "begin_time_s", "end_time_s", "selection"]
    )
    .reset_index(drop=True)
)

# Sequential event number within each recording
master_events["event_number"] = (
    master_events.groupby("itemID").cumcount() + 1
)

# Globally unique event identifier
master_events["event_id"] = (
    master_events["itemID"].astype(str)
    + "_E"
    + master_events["event_number"].astype(str).str.zfill(4)
)

print(f"Total events:     {len(master_events):,}")
print(f"Unique event IDs: {master_events['event_id'].nunique():,}")
print(
    "Duplicate event IDs:",
    master_events["event_id"].duplicated().sum()
)

Total events:     184,086
Unique event IDs: 184,086
Duplicate event IDs: 0


In [35]:
# ============================================================
# 25. Refresh Recording Event Counts
# ============================================================

event_counts = (
    master_events
    .groupby("itemID")
    .size()
    .rename("n_events")
)

master_recordings = (
    recordings
    .drop(columns=["n_events", "has_events"], errors="ignore")
    .merge(
        event_counts,
        on="itemID",
        how="left"
    )
)

master_recordings["n_events"] = (
    master_recordings["n_events"]
    .fillna(0)
    .astype(int)
)

master_recordings["has_events"] = (
    master_recordings["n_events"] > 0
)

print(f"Recordings:             {len(master_recordings):,}")
print(f"Recordings with events: {master_recordings['has_events'].sum():,}")
print(f"Zero-event recordings:  {(~master_recordings['has_events']).sum():,}")
print(f"Total events:           {master_recordings['n_events'].sum():,}")

Recordings:             1,302
Recordings with events: 1,295
Zero-event recordings:  7
Total events:           184,086


In [36]:
# ============================================================
# 26. Save Canonical Cleaned Tables
# ============================================================

DERIVED_DIR = DATA_DIR / "derived"
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

events_path = DERIVED_DIR / "master_events.csv"
recordings_path = DERIVED_DIR / "master_recordings.csv"

master_events.to_csv(events_path, index=False)
master_recordings.to_csv(recordings_path, index=False)

print(f"Saved events to:\n  {events_path}")
print(f"\nSaved recordings to:\n  {recordings_path}")

print("\nFinal dataset:")
print(f"  Recordings:             {len(master_recordings):,}")
print(f"  Recordings with events: {master_recordings['has_events'].sum():,}")
print(f"  Zero-event recordings:  {(~master_recordings['has_events']).sum():,}")
print(f"  Unique events:          {len(master_events):,}")
print(f"  Annotation codes:       {master_events['species_code'].nunique():,}")

Saved events to:
  /content/drive/MyDrive/Bioacoustics_Project/data/derived/master_events.csv

Saved recordings to:
  /content/drive/MyDrive/Bioacoustics_Project/data/derived/master_recordings.csv

Final dataset:
  Recordings:             1,302
  Recordings with events: 1,295
  Zero-event recordings:  7
  Unique events:          184,086
  Annotation codes:       101


### Data integration and quality-control summary

The three annotation datasets were integrated with recording-, site-, and
species-level metadata using `ItemID` as the authoritative recording identifier.

The complete dataset contains:

- **1,302 recordings**
- **1,295 recordings with ≥1 annotated event**
- **7 recordings with zero annotated events**
- **184,086 unique annotated vocalization events**
- **101 standardized annotation codes**

Quality-control checks identified:

1. **Eight DatasetSIMR filename discrepancies.** The recording filename stored
   in the Raven annotation did not exactly match the filename in the recording
   metadata. All eight recordings were unambiguously reconciled using `ItemID`.

2. **Three species-code formatting errors.** `HETH`, `HAWO`, and `WOTH`
   occurred with leading or trailing whitespace. Species codes were standardized
   by removing leading/trailing whitespace.

3. **121 exact duplicate annotations in one recording (ItemID 6017).**
   These records were identical in selection number, species identity, time
   bounds, and frequency bounds. One copy of each duplicate was retained,
   reducing the event table from 184,207 to 184,086 unique events.

The original annotation files and metadata remain unchanged. All corrections
were applied only to the derived analysis tables.

Each event has a globally unique `event_id` and retains its original Raven
selection number and source-file information for provenance.